# Score Stability Experiments

这个 notebook 准备了两个实验：

1. 同一个 PDF 连续跑 10 次，统计总分和各 section 分数的方差。
2. 两组 PDF 各 5 个，统计两组分布并做可视化比较。

请把文件放到下面这些目录：

- `data/experiments/same_pdf/`：放 1 个 PDF，用于实验 1
- `data/experiments/group_a/`：放第 1 组 PDF，建议 5 个
- `data/experiments/group_b/`：放第 2 组 PDF，建议 5 个
- `data/experiments/results/`：运行结果会自动写到这里

默认使用当前项目里的 `qwen3_ollama.py` 评分链路。

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from qwen3_ollama import _Scorer, score_application
from src.all_type_parser.all_type_parser import parse_and_save

CRITERIA_PATH = PROJECT_ROOT / 'criteria_points.json'
EXPERIMENT_ROOT = PROJECT_ROOT / 'data' / 'experiments'
SAME_PDF_DIR = EXPERIMENT_ROOT / 'same_pdf'
GROUP_A_DIR = EXPERIMENT_ROOT / 'group_a'
GROUP_B_DIR = EXPERIMENT_ROOT / 'group_b'
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
PARSED_CACHE_DIR = RESULTS_DIR / 'parsed_cache'

for path in [SAME_PDF_DIR, GROUP_A_DIR, GROUP_B_DIR, RESULTS_DIR, PARSED_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SECTION_KEYS = [
    'general',
    'proposed_research',
    'training_development',
    'sites_support',
    'wpcc',
    'application_form',
]

print('Project root:', PROJECT_ROOT)
print('Same PDF dir:', SAME_PDF_DIR)
print('Group A dir:', GROUP_A_DIR)
print('Group B dir:', GROUP_B_DIR)
print('Results dir:', RESULTS_DIR)


In [ ]:
def list_pdfs(folder: Path) -> list[Path]:
    return sorted([path for path in folder.glob('*.pdf') if path.is_file()])


def parse_pdf_cached(pdf_path: Path, *, reparse: bool = False) -> tuple[dict, Path]:
    json_path = PARSED_CACHE_DIR / f'{pdf_path.stem}.json'
    if reparse or not json_path.exists():
        parse_and_save(str(pdf_path), str(json_path))
    parsed = json.loads(json_path.read_text(encoding='utf-8'))
    return parsed, json_path


def score_pdf_once(
    pdf_path: Path,
    *,
    scorer: _Scorer,
    run_tag: str,
    reparse: bool = False,
) -> dict:
    parsed, parsed_json_path = parse_pdf_cached(pdf_path, reparse=reparse)
    artifact_dir = RESULTS_DIR / run_tag
    artifact_dir.mkdir(parents=True, exist_ok=True)
    result = score_application(
        parsed,
        CRITERIA_PATH,
        doc_id=f'{pdf_path.stem}_{run_tag}',
        scorer=scorer,
        artifacts_dir=artifact_dir,
    )
    result['source_pdf'] = str(pdf_path)
    result['parsed_json'] = str(parsed_json_path)
    out_path = artifact_dir / f'{pdf_path.stem}_{run_tag}_scored.json'
    out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    result['result_json'] = str(out_path)
    return result


def flatten_result_row(result: dict, *, pdf_name: str, run_idx: int | None, group: str | None) -> dict:
    row = {
        'pdf_name': pdf_name,
        'run_idx': run_idx,
        'group': group,
        'overall_score_100': float(result.get('overall', {}).get('final_score_0to100', 0)),
        'overall_score_10': float(result.get('overall', {}).get('score_10', 0)),
        'avg_plausibility_0to5': float(result.get('overall', {}).get('avg_plausibility_0to5', 0)),
        'weak_signal_count': int(result.get('overall', {}).get('weak_signal_count', 0)),
        'pool_size': int(result.get('pool_size', 0)),
        'source_pdf': result.get('source_pdf'),
        'parsed_json': result.get('parsed_json'),
        'result_json': result.get('result_json'),
    }
    features = result.get('features', {})
    for section_key in SECTION_KEYS:
        section = features.get(section_key, {})
        overall = section.get('overall', {})
        row[f'{section_key}_score_100'] = float(overall.get('final_score_0to100', 0))
        row[f'{section_key}_score_10'] = float(overall.get('score_10', 0))
        row[f'{section_key}_avg_plausibility'] = float(overall.get('avg_plausibility_0to5', 0))
        row[f'{section_key}_weak_signal_count'] = int(overall.get('weak_signal_count', 0))
    return row


def score_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in df.columns if col.endswith('_score_100') or col == 'overall_score_100']


def summarize_numeric(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    summary = pd.DataFrame({
        'mean': df[columns].mean(),
        'std': df[columns].std(ddof=1),
        'var': df[columns].var(ddof=1),
        'min': df[columns].min(),
        'max': df[columns].max(),
        'median': df[columns].median(),
    }).reset_index().rename(columns={'index': 'metric'})
    return summary


def plot_same_pdf(df: pd.DataFrame, pdf_name: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(df['run_idx'], df['overall_score_100'], marker='o')
    axes[0].set_title(f'Overall score by run: {pdf_name}')
    axes[0].set_xlabel('Run index')
    axes[0].set_ylabel('Score / 100')
    axes[0].grid(alpha=0.3)

    axes[1].boxplot(df['overall_score_100'], vert=True)
    axes[1].set_title('Overall score distribution')
    axes[1].set_ylabel('Score / 100')
    plt.tight_layout()
    plt.show()


def plot_group_compare(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for group_name, group_df in df.groupby('group'):
        axes[0].hist(group_df['overall_score_100'], bins=min(10, max(3, len(group_df))), alpha=0.5, label=group_name)
    axes[0].set_title('Overall score distribution by group')
    axes[0].set_xlabel('Score / 100')
    axes[0].set_ylabel('Count')
    axes[0].legend()

    grouped = [group_df['overall_score_100'].tolist() for _, group_df in sorted(df.groupby('group'))]
    labels = [group_name for group_name, _ in sorted(df.groupby('group'))]
    axes[1].boxplot(grouped, tick_labels=labels)
    axes[1].set_title('Overall score boxplot')
    axes[1].set_ylabel('Score / 100')
    plt.tight_layout()
    plt.show()


def optional_stat_tests(group_a_scores: pd.Series, group_b_scores: pd.Series) -> pd.DataFrame:
    rows = []
    try:
        from scipy.stats import ks_2samp, ttest_ind
        t_stat, t_p = ttest_ind(group_a_scores, group_b_scores, equal_var=False)
        ks_stat, ks_p = ks_2samp(group_a_scores, group_b_scores)
        rows.append({'test': 'welch_ttest', 'statistic': float(t_stat), 'p_value': float(t_p)})
        rows.append({'test': 'ks_2samp', 'statistic': float(ks_stat), 'p_value': float(ks_p)})
    except Exception as exc:
        rows.append({'test': 'scipy_unavailable', 'statistic': None, 'p_value': None, 'note': str(exc)})
    return pd.DataFrame(rows)


def run_same_pdf_experiment(
    pdf_path: Path,
    *,
    n_runs: int = 10,
    reparse_each_run: bool = False,
    sleep_seconds: float = 0.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    scorer = _Scorer()
    rows = []
    for run_idx in range(1, n_runs + 1):
        run_tag = f'same_pdf_run_{run_idx:02d}'
        result = score_pdf_once(
            pdf_path,
            scorer=scorer,
            run_tag=run_tag,
            reparse=reparse_each_run,
        )
        rows.append(flatten_result_row(result, pdf_name=pdf_path.name, run_idx=run_idx, group='same_pdf'))
        if sleep_seconds > 0:
            time.sleep(sleep_seconds)
    df = pd.DataFrame(rows)
    summary = summarize_numeric(df, score_columns(df))
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    df.to_csv(RESULTS_DIR / f'same_pdf_runs_{timestamp}.csv', index=False)
    summary.to_csv(RESULTS_DIR / f'same_pdf_summary_{timestamp}.csv', index=False)
    return df, summary


def run_group_compare_experiment(
    *,
    group_a_paths: list[Path],
    group_b_paths: list[Path],
    runs_per_pdf: int = 1,
    reparse_each_run: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    scorer = _Scorer()
    rows = []
    for group_name, pdf_paths in [('group_a', group_a_paths), ('group_b', group_b_paths)]:
        for pdf_path in pdf_paths:
            for run_idx in range(1, runs_per_pdf + 1):
                run_tag = f'{group_name}_{pdf_path.stem}_run_{run_idx:02d}'
                result = score_pdf_once(
                    pdf_path,
                    scorer=scorer,
                    run_tag=run_tag,
                    reparse=reparse_each_run,
                )
                rows.append(flatten_result_row(result, pdf_name=pdf_path.name, run_idx=run_idx, group=group_name))
    df = pd.DataFrame(rows)
    score_cols = score_columns(df)
    summary = df.groupby('group')[score_cols].agg(['mean', 'std', 'var', 'min', 'max', 'median'])
    tests = optional_stat_tests(
        df.loc[df['group'] == 'group_a', 'overall_score_100'],
        df.loc[df['group'] == 'group_b', 'overall_score_100'],
    )
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    df.to_csv(RESULTS_DIR / f'group_compare_runs_{timestamp}.csv', index=False)
    summary.to_csv(RESULTS_DIR / f'group_compare_summary_{timestamp}.csv')
    tests.to_csv(RESULTS_DIR / f'group_compare_tests_{timestamp}.csv', index=False)
    return df, summary, tests


## 实验 1：同一个 PDF 跑 10 次

把 1 个 PDF 放到 `data/experiments/same_pdf/`，然后运行下面这个 cell。

In [ ]:
same_pdf_files = list_pdfs(SAME_PDF_DIR)
assert len(same_pdf_files) == 1, f'请在 {SAME_PDF_DIR} 里放且只放 1 个 PDF，当前有 {len(same_pdf_files)} 个。'

same_pdf_path = same_pdf_files[0]
same_pdf_df, same_pdf_summary = run_same_pdf_experiment(
    same_pdf_path,
    n_runs=10,
    reparse_each_run=False,
    sleep_seconds=0.0,
)

display(same_pdf_df[['run_idx', 'overall_score_100', 'avg_plausibility_0to5', 'weak_signal_count']])
display(same_pdf_summary)
plot_same_pdf(same_pdf_df, same_pdf_path.name)


## 实验 2：两组 PDF 分布比较

把两组 PDF 分别放到：

- `data/experiments/group_a/`
- `data/experiments/group_b/`

建议每组 5 个。默认每个 PDF 跑 1 次。

In [ ]:
group_a_files = list_pdfs(GROUP_A_DIR)
group_b_files = list_pdfs(GROUP_B_DIR)

assert len(group_a_files) == 5, f'请在 {GROUP_A_DIR} 里放 5 个 PDF，当前有 {len(group_a_files)} 个。'
assert len(group_b_files) == 5, f'请在 {GROUP_B_DIR} 里放 5 个 PDF，当前有 {len(group_b_files)} 个。'

group_df, group_summary, group_tests = run_group_compare_experiment(
    group_a_paths=group_a_files,
    group_b_paths=group_b_files,
    runs_per_pdf=1,
    reparse_each_run=False,
)

display(group_df[['group', 'pdf_name', 'overall_score_100', 'avg_plausibility_0to5', 'weak_signal_count']])
display(group_summary)
display(group_tests)
plot_group_compare(group_df)
